In [122]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [19]:
train_df  = pd.read_csv("train_play.csv")
display(train_df.head())
print(train_df.shape)

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


(2190, 13)


In [33]:
test_df = pd.read_csv("test_play.csv")
display(test_df.head())
print(test_df.shape)

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4


(730, 12)


In [145]:
features_data = train_df.drop(columns=["rainfall", "id"])
print("Features are :",features)
print(features_data.shape)
y = train_df["rainfall"]

Features are : ['day', 'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed']
(2190, 11)


In [71]:
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import xgboost
import catboost
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
print("Using XGBoost version",xgboost.__version__)
print("Using CatBoost version",catboost.__version__)

Using XGBoost version 2.1.3
Using CatBoost version 1.2.7


In [203]:
fold = 15
skf = StratifiedKFold(n_splits=fold,shuffle=True,random_state=700)

auc_scores = []

for fold, (train_index, valid_index) in enumerate(skf.split(features_data, y), start=1):
    print("#" * 25)
    print(f"### Fold {fold}")
    print("#" * 25)
    
    features_train = features_data.iloc[train_index].copy()
    y_train = y.iloc[train_index]    
    features_valid = features_data.iloc[valid_index].copy()
    y_valid = y.iloc[valid_index]

    xgb = XGBClassifier(
        device="cuda",
        max_depth=5,  
        colsample_bytree=0.9, 
        subsample=0.9, 
        n_estimators=10_000,  
        learning_rate=0.9, 
        eval_metric="auc",
        verbosity=0
    )

    xgb.fit(features_train,y_train)

    y_valid_proba = xgb.predict_proba(features_valid)[:,1]

    auc = roc_auc_score(y_valid,y_valid_proba)
    auc_scores.append(auc)

    print(f"fold {fold}  XGB: AUC = {auc:.4f}")
    #print(f"mean of auc for xgb is : {np.mean(auc):.4f}")

#########################
### Fold 1
#########################
fold 1  XGB: AUC = 0.8071
#########################
### Fold 2
#########################
fold 2  XGB: AUC = 0.8861
#########################
### Fold 3
#########################
fold 3  XGB: AUC = 0.8667
#########################
### Fold 4
#########################
fold 4  XGB: AUC = 0.7883
#########################
### Fold 5
#########################
fold 5  XGB: AUC = 0.8083
#########################
### Fold 6
#########################
fold 6  XGB: AUC = 0.8760
#########################
### Fold 7
#########################
fold 7  XGB: AUC = 0.8614
#########################
### Fold 8
#########################
fold 8  XGB: AUC = 0.8816
#########################
### Fold 9
#########################
fold 9  XGB: AUC = 0.8461
#########################
### Fold 10
#########################
fold 10  XGB: AUC = 0.8456
#########################
### Fold 11
#########################
fold 11  XGB: AUC = 0.9183
#################

In [204]:
for fold, (train_index, valid_index) in enumerate(skf.split(features_data, y), start=1):
    print("#" * 25)
    print(f"### Fold {fold}")
    print("#" * 25)
    
    features_train = features_data.iloc[train_index].copy()
    y_train = y.iloc[train_index]    
    features_valid = features_data.iloc[valid_index].copy()
    y_valid = y.iloc[valid_index]

    cat = CatBoostClassifier(
        max_depth=5,  
        random_seed=420, 
        subsample=0.9, 
        n_estimators=10_000,  
        learning_rate=0.09, 
        eval_metric="AUC",
        verbose=0
    )

    cat.fit(features_train,y_train)

    y_valid_proba = cat.predict_proba(features_valid)[:,1]

    auc_cat = roc_auc_score(y_valid,y_valid_proba)
    auc_scores.append(auc_cat)

    print(f"fold {fold}  CAT: AUC = {auc_cat:.4f}")

    #print(f"mean of auc for cat is : {np.mean(auc_cat):.4f}")

#########################
### Fold 1
#########################
fold 1  CAT: AUC = 0.8394
#########################
### Fold 2
#########################
fold 2  CAT: AUC = 0.9104
#########################
### Fold 3
#########################
fold 3  CAT: AUC = 0.8672
#########################
### Fold 4
#########################
fold 4  CAT: AUC = 0.8298
#########################
### Fold 5
#########################
fold 5  CAT: AUC = 0.8510
#########################
### Fold 6
#########################
fold 6  CAT: AUC = 0.8894
#########################
### Fold 7
#########################
fold 7  CAT: AUC = 0.8851
#########################
### Fold 8
#########################
fold 8  CAT: AUC = 0.9285
#########################
### Fold 9
#########################
fold 9  CAT: AUC = 0.8010
#########################
### Fold 10
#########################
fold 10  CAT: AUC = 0.8566
#########################
### Fold 11
#########################
fold 11  CAT: AUC = 0.9230
#################

In [206]:
test_df1 = test_df.copy()
features = train_df.drop(columns=["rainfall", "id"], axis=1).columns.tolist()
test_df1=test_df1[features]
y_pred_test = cat.predict_proba(test_df1)[:,1]

In [207]:
rainfall_df = pd.DataFrame({"id":test_df["id"],"rainfall":y_pred_test})
rainfall_df
rainfall_df.to_csv("submission_rain.csv",index=False)